In [1]:
import sys
sys.path.append('..')
from config import DB_USER, DB_PASSWORD, DB_HOST, DB_PORT, DB_NAME
from sqlalchemy import create_engine
import pandas as pd

engine = create_engine(
    f"mysql+mysqlconnector://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

query = """
SELECT 
    r.grid_position,
    r.finish_position,
    r.points,
    d.driver_name,
    ra.season,
    ra.round,
    ra.circuit_name
FROM results r
JOIN drivers d ON r.driver_id = d.driver_id
JOIN races ra ON r.race_id = ra.race_id
"""

df = pd.read_sql(query, con=engine)
print(df.shape)
df.head()

(479, 7)


,grid_position,finish_position,points,driver_name,season,round,circuit_name
0,1,1,25.0,Lando Norris,2025,1,Australian Grand Prix
1,3,2,18.0,Lando Norris,2025,2,Chinese Grand Prix
2,2,2,18.0,Lando Norris,2025,3,Japanese Grand Prix
3,6,3,15.0,Lando Norris,2025,4,Bahrain Grand Prix
4,10,4,12.0,Lando Norris,2025,5,Saudi Arabian Grand Prix


In [2]:
X = df[['grid_position']]
y = df['finish_position']

print(X.shape, y.shape)

(479, 1) (479,)


In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training rows: {len(X_train)}")
print(f"Test rows: {len(X_test)}")

Training rows: 383
Test rows: 96


In [5]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train, y_train)

print(f"Slope (coefficient): {model.coef_[0]:.3f}")
print(f"Intercept: {model.intercept_:.3f}")

Slope (coefficient): 0.672
Intercept: 3.440


In [6]:
from sklearn.metrics import mean_absolute_error, r2_score

y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Absolute Error: {mae:.2f} positions")
print(f"R² score: {r2:.3f}")

Mean Absolute Error: 3.76 positions
R² score: 0.297


In [7]:
import numpy as np

baseline_pred = X_test['grid_position'].values

baseline_mae = mean_absolute_error(y_test, baseline_pred)
baseline_r2 = r2_score(y_test, baseline_pred)

print(f"Baseline (finish = grid):")
print(f"  MAE: {baseline_mae:.2f} positions")
print(f"  R²: {baseline_r2:.3f}")
print()
print(f"Your model:")
print(f"  MAE: {mae:.2f} positions")
print(f"  R²: {r2:.3f}")

Baseline (finish = grid):
  MAE: 3.86 positions
  R²: 0.127

Your model:
  MAE: 3.76 positions
  R²: 0.297


In [8]:
results_summary = pd.DataFrame({
    'model': ['Baseline (finish=grid)', 'Linear Regression'],
    'mae': [baseline_mae, mae],
    'r2': [baseline_r2, r2]
})

results_summary.to_csv('../data/model_results.csv', index=False)
results_summary

,model,mae,r2
0,Baseline (finish=grid),3.864583,0.126740
1,Linear Regression,3.756793,0.296582


In [9]:
csv_df = pd.read_csv('../data/f1_2025_results.csv')

df2 = df.merge(
    csv_df[['season', 'round', 'driver_name', 'constructor']],
    on=['season', 'round', 'driver_name'],
    how='left'
)

print(df2['constructor'].isnull().sum())
df2.head()

0


,grid_position,finish_position,points,driver_name,season,round,circuit_name,constructor
0,1,1,25.0,Lando Norris,2025,1,Australian Grand Prix,McLaren
1,3,2,18.0,Lando Norris,2025,2,Chinese Grand Prix,McLaren
2,2,2,18.0,Lando Norris,2025,3,Japanese Grand Prix,McLaren
3,6,3,15.0,Lando Norris,2025,4,Bahrain Grand Prix,McLaren
4,10,4,12.0,Lando Norris,2025,5,Saudi Arabian Grand Prix,McLaren


In [10]:
constructor_dummies = pd.get_dummies(df2['constructor'], prefix='team')

X2 = pd.concat([df2[['grid_position']], constructor_dummies], axis=1)
y2 = df2['finish_position']

print(f"Features now: {X2.shape[1]}")
X2.head()

Features now: 11


,grid_position,team_Alpine F1 Team,team_Aston Martin,team_Ferrari,team_Haas F1 Team,team_McLaren,team_Mercedes,team_RB F1 Team,team_Red Bull,team_Sauber,team_Williams
0,1,False,False,False,False,True,False,False,False,False,False
1,3,False,False,False,False,True,False,False,False,False,False
2,2,False,False,False,False,True,False,False,False,False,False
3,6,False,False,False,False,True,False,False,False,False,False
4,10,False,False,False,False,True,False,False,False,False,False


In [11]:
X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2, test_size=0.2, random_state=42
)

model2 = LinearRegression()
model2.fit(X2_train, y2_train)

y2_pred = model2.predict(X2_test)

mae2 = mean_absolute_error(y2_test, y2_pred)
r2_2 = r2_score(y2_test, y2_pred)

print(f"Grid only:        MAE {mae:.2f}, R² {r2:.3f}")
print(f"Grid + team:      MAE {mae2:.2f}, R² {r2_2:.3f}")

Grid only:        MAE 3.76, R² 0.297
Grid + team:      MAE 3.45, R² 0.341


In [12]:
coef_df = pd.DataFrame({
    'feature': X2.columns,
    'coefficient': model2.coef_
}).sort_values('coefficient')

coef_df

,feature,coefficient
5,team_McLaren,-2.170868
6,team_Mercedes,-1.561438
3,team_Ferrari,-1.361370
8,team_Red Bull,-1.184603
4,team_Haas F1 Team,-0.439350
10,team_Williams,-0.233049
0,grid_position,0.531399
2,team_Aston Martin,1.025701
9,team_Sauber,1.469016
7,team_RB F1 Team,1.737468


In [13]:
tableau_export = df2.copy()
tableau_export['predicted_finish'] = model2.predict(X2)
tableau_export['position_change'] = tableau_export['grid_position'] - tableau_export['finish_position']
tableau_export['prediction_error'] = tableau_export['finish_position'] - tableau_export['predicted_finish']

tableau_export.to_csv('../data/tableau_export.csv', index=False)
print(tableau_export.shape)

(479, 11)


In [18]:
readme = """# F1 Race Outcome Predictor

Predicting Formula 1 finishing positions from qualifying grid position using Python, MySQL, and machine learning.

**[View the interactive dashboard on Tableau Public](https://public.tableau.com/views/F1RacePredictor/Dashboard1)**

## Project overview

This project explores how much a driver's starting grid position determines their race result, using 2025 Formula 1 season data (479 results across 24 rounds).

### Key findings

- Grid position correlates **0.651** with finishing position, explaining roughly 42% of variance across the full dataset.
- A linear regression using grid position alone achieved **R2 0.297** on held-out test data, with a mean absolute error of **3.76 positions**.
- Adding constructor (team) as a feature improved this to **R2 0.341** and **MAE 3.45** - confirming that car pace carries information grid position alone doesn't capture.
- The model marginally beat a naive baseline (predicting finish = grid) on MAE, but nearly tripled its R2, indicating it captures the underlying relationship better even where individual predictions remain imprecise.

**Interpretation:** grid position is a real but weak predictor. F1 race outcomes are substantially driven by factors outside qualifying performance - reliability, strategy, incidents, and race-day pace.

## Tech stack

- **Python** (pandas, matplotlib, scikit-learn) - extraction, analysis, modelling
- **MySQL** - normalized relational storage
- **Jupyter** - exploratory analysis
- **Tableau Public** - interactive dashboard
- **n8n** - automated data pipeline
- **Docker** - running n8n locally
- **Git / GitHub** - version control

## Data source

[jolpica-f1 API](https://api.jolpi.ca/) - the community-maintained successor to the deprecated Ergast API. Free, no authentication required.

## Database schema

Three normalized tables:

- `drivers` - one row per driver (21 in 2025)
- `races` - one row per Grand Prix (24 rounds)
- `results` - one row per driver per race, linked to both via foreign keys

## Repository structure

- `data/` - raw and processed datasets
- `notebooks/` - Jupyter notebooks (01_data_extraction, 02_modelling)
- `sql/` - SQL queries and schema scripts
- `scripts/` - standalone Python scripts

## Automation (n8n)

A scheduled n8n workflow keeps the pipeline running without manual intervention:

1. **Schedule Trigger** - fires weekly (Mondays, 9am)
2. **HTTP Request** - fetches the most recent completed race from the jolpica API using the `/current/last/` endpoint
3. **Code node** (JavaScript) - extracts driver, constructor, grid and finish position from the nested JSON
4. **Convert to File** - outputs as CSV

## Known limitations

### Data and schema

- `races.circuit_name` stores the race name (e.g. "Australian Grand Prix") rather than the circuit (e.g. "Albert Park"). These should be separate columns.
- `races.race_date` is not populated, though the API provides it.
- No `constructors` table - team data is joined from CSV rather than stored relationally.
- Constructor naming is inconsistent across seasons (e.g. "RB F1 Team" vs "Racing Bulls"). Entity resolution would be needed before combining multiple seasons.

### Analysis

- The position-change metric partly reflects where teams qualify, since gaining places is easier from the back of the grid.
- Only 2025 season data was used for modelling; more seasons would give a more robust model.

### Automation

- n8n runs in a Docker container and cannot reach the local MySQL instance without additional network configuration, so output is written to file rather than inserted directly into the database.
- The n8n Python runner is unavailable in the standard Docker image; the transform node uses JavaScript instead.
- The schedule only fires while the container is running. Production deployment would require hosting n8n on a persistent server.
- The pipeline collects current-season (2026) data while the model is trained on 2025. Reconciling the two - retraining across multiple seasons and handling constructor renames - is the next development step.

## Setup

Requires a local MySQL instance and a `config.py` file in the project root (gitignored) containing database credentials.

## Progress

- [x] Data extraction from API
- [x] MySQL schema design and ETL pipeline
- [x] Exploratory data analysis
- [x] Baseline and improved prediction models
- [x] Tableau dashboard
- [x] n8n automated pipeline
- [ ] Multi-season data and model retraining
- [ ] Forward prediction: predict upcoming races before they run
"""

with open('../README.md', 'w') as f:
    f.write(readme)

print("README updated")

README updated
